<a href="https://colab.research.google.com/github/DeemonDuck/upi-sentinel/blob/main/inference_engine_raw_transactions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#Imports
import tensorflow as tf
import numpy as np
import pandas as pd
import joblib
import json

print("Libraries imported successfully.")

Libraries imported successfully.


In [ ]:
#Calling Artificats and Other Data

SAVE_PATH = "/content/drive/MyDrive/UPI_SENTINEL/COLAB_NOTEBOOKS/Colab_File_Saved_Checkpoints/"

print("Artifact path set successfully.")

# Load scaler
scaler = joblib.load(SAVE_PATH + "scaler.save")

# Load label encoder
label_encoder = joblib.load(SAVE_PATH + "label_encoder.save")

# Load metadata
with open(SAVE_PATH + "metadata.json", "r") as f:
    metadata = json.load(f)

SEQUENCE_LENGTH = metadata["sequence_length"]
THRESHOLD = metadata["threshold"]

print("Preprocessing artifacts loaded.")
print("Sequence Length:", SEQUENCE_LENGTH)
print("Fraud Threshold:", THRESHOLD)

Artifact path set successfully.
Preprocessing artifacts loaded.
Sequence Length: 5
Fraud Threshold: 0.99532


In [ ]:
#Attention Layer From Artificats

import sys
sys.path.append(SAVE_PATH)
from attention_layer import AttentionLayer
print("AttentionLayer imported successfully")

AttentionLayer imported successfully


In [ ]:
#Loading Model

model = tf.keras.models.load_model(
    SAVE_PATH + "best_model.keras",
    custom_objects={"AttentionLayer": AttentionLayer}
)
print("Model loaded successfully")

Model loaded successfully


/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_lib.py:802: UserWarning: Skipping variable loading for optimizer 'adam', because it has 32 variables whereas the saved optimizer has 2 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


# RAW INPUT PROCESSING

In [ ]:
def preprocess_transaction(txn):

    df = pd.DataFrame([txn])

    # Encode transaction type
    df['type'] = label_encoder.transform(df['type'])

    # Numeric columns
    num_cols = [
        'amount',
        'oldbalanceOrg',
        'newbalanceOrig',
        'oldbalanceDest',
        'newbalanceDest'
    ]

    df[num_cols] = scaler.transform(df[num_cols])

    # Select final feature order
    features = [
        'type',
        'amount',
        'oldbalanceOrg',
        'newbalanceOrig',
        'oldbalanceDest',
        'newbalanceDest'
    ]

    return df[features].values

#SLIDING WINDOW ENGINE

## Sliding Window Buffer

In [ ]:
#Buffer to store last 5 transaction
transaction_buffer = []

## Sliding Window Logic

In [ ]:
def update_sequence(txn):

    transaction_buffer.append(txn[0])

    if len(transaction_buffer) > SEQUENCE_LENGTH:
        transaction_buffer.pop(0)

    return np.array(transaction_buffer)

## Prediction Engine

In [ ]:
def predict_transaction(txn):

    processed = preprocess_transaction(txn)

    seq = update_sequence(processed)

    if len(seq) < SEQUENCE_LENGTH:
        return {"message": "Waiting for more transactions"}

    seq = seq.reshape(1, SEQUENCE_LENGTH, 6)

    prob = model.predict(seq, verbose=0)[0][0]

    if prob < 0.5:
        risk = "SAFE"
    elif prob < 0.9:
        risk = "SUSPICIOUS"
    else:
        risk = "FRAUD"

    return {
        "fraud_probability": float(prob),
        "risk_level": risk
    }

##Stream Simulation

In [ ]:
def simulate_transaction_stream(transactions):

    print("Starting Transaction Stream...\n")

    for i, txn in enumerate(transactions, start=1):

        result = predict_transaction(txn)

        print(f"Transaction {i}")
        print("Input:", txn)
        print("Output:", result)
        print("-" * 50)

Test the System

In [ ]:
transactions = [

{
"step": 1,
"type": "TRANSFER",
"amount": 2000,
"oldbalanceOrg": 5000,
"newbalanceOrig": 3000,
"oldbalanceDest": 1000,
"newbalanceDest": 3000
},

{
"step": 2,
"type": "TRANSFER",
"amount": 1500,
"oldbalanceOrg": 3000,
"newbalanceOrig": 1500,
"oldbalanceDest": 3000,
"newbalanceDest": 4500
},

{
"step": 3,
"type": "TRANSFER",
"amount": 1200,
"oldbalanceOrg": 1500,
"newbalanceOrig": 300,
"oldbalanceDest": 4500,
"newbalanceDest": 5700
},

{
"step": 4,
"type": "TRANSFER",
"amount": 250,
"oldbalanceOrg": 300,
"newbalanceOrig": 50,
"oldbalanceDest": 5700,
"newbalanceDest": 5950
},

{
"step": 5,
"type": "TRANSFER",
"amount": 50,
"oldbalanceOrg": 50,
"newbalanceOrig": 0,
"oldbalanceDest": 5950,
"newbalanceDest": 6000
}

# {
# "step": 5,
# "type": "PAYMENT",
# "amount": 10,
# "oldbalanceOrg": 50,
# "newbalanceOrig": 40,
# "oldbalanceDest": 5950,
# "newbalanceDest": 5960
# }
]

In [ ]:
simulate_transaction_stream(transactions)

Starting Transaction Stream...

Transaction 1
Input: {'step': 1, 'type': 'TRANSFER', 'amount': 2000, 'oldbalanceOrg': 5000, 'newbalanceOrig': 3000, 'oldbalanceDest': 1000, 'newbalanceDest': 3000}
Output: {'message': 'Waiting for more transactions'}
--------------------------------------------------
Transaction 2
Input: {'step': 2, 'type': 'TRANSFER', 'amount': 1500, 'oldbalanceOrg': 3000, 'newbalanceOrig': 1500, 'oldbalanceDest': 3000, 'newbalanceDest': 4500}
Output: {'message': 'Waiting for more transactions'}
--------------------------------------------------
Transaction 3
Input: {'step': 3, 'type': 'TRANSFER', 'amount': 1200, 'oldbalanceOrg': 1500, 'newbalanceOrig': 300, 'oldbalanceDest': 4500, 'newbalanceDest': 5700}
Output: {'message': 'Waiting for more transactions'}
--------------------------------------------------
Transaction 4
Input: {'step': 4, 'type': 'TRANSFER', 'amount': 250, 'oldbalanceOrg': 300, 'newbalanceOrig': 50, 'oldbalanceDest': 5700, 'newbalanceDest': 5950}
Outpu

In [ ]:
transactions = [{"step": 1, "type": "TRANSFER", "amount": 2000, "oldbalanceOrg": 5000, "newbalanceOrig": 3000, "oldbalanceDest": 1000, "newbalanceDest": 3000}, {"step": 2, "type": "TRANSFER", "amount": 1500, "oldbalanceOrg": 3000, "newbalanceOrig": 1500, "oldbalanceDest": 3000, "newbalanceDest": 4500}, {"step": 3, "type": "TRANSFER", "amount": 1200, "oldbalanceOrg": 1500, "newbalanceOrig": 300, "oldbalanceDest": 4500, "newbalanceDest": 5700}, {"step": 4, "type": "TRANSFER", "amount": 250, "oldbalanceOrg": 300, "newbalanceOrig": 50, "oldbalanceDest": 5700, "newbalanceDest": 5950}, {"step": 5, "type": "TRANSFER", "amount": 50, "oldbalanceOrg": 50, "newbalanceOrig": 0, "oldbalanceDest": 5950, "newbalanceDest": 6000}]